In [1]:
import re
import os
import fsspec, s3fs
import datasets
import sys
sys.path.insert(0, "/home/sagemaker-user/assignment5-alignment/verl")

from verl.utils.hdfs_io import copy, makedirs
import argparse

from multiprocessing import set_start_method
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
# **Must** happen before torch or vllm ever touches CUDA
set_start_method("spawn", force=True)

from datasets import interleave_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedTokenizerFast, PreTrainedModel
from transformers import Trainer, TrainingArguments
from trl import SFTConfig, SFTTrainer
from trl import setup_chat_format
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from datasets import load_dataset, Dataset
from concurrent.futures import ThreadPoolExecutor
from trl import DataCollatorForCompletionOnlyLM
import torch
from vllm import LLM, SamplingParams
from vllm.model_executor import set_random_seed as vllm_set_random_seed
from drgrpo_grader import r1_zero_reward_fn
import gc
from unittest.mock import patch
import wandb
import safetensors
import os
import json
import numpy as np
import time
import random 
import operator
from transformers import TrainerCallback, TrainerState, TrainerControl

/home/sagemaker-user/assignment5-alignment/verl/verl/__init__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


INFO 06-30 04:05:00 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 06-30 04:05:00 [__init__.py:239] Automatically detected platform cuda.


In [1]:
import torch, sysconfig
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CXXFLAGS:", sysconfig.get_config_var("CXXFLAGS"))

import sysconfig
print("CFLAGS:      ", sysconfig.get_config_var("CFLAGS"))
print("BASECFLAGS:  ", sysconfig.get_config_var("BASECFLAGS"))
print("OPT:         ", sysconfig.get_config_var("OPT"))
print("CONFIG_ARGS: ", sysconfig.get_config_var("CONFIG_ARGS"))


Torch: 2.6.0+cu124
Torch CUDA: 12.4
CXXFLAGS: None
CFLAGS:       -fno-strict-overflow -Wsign-compare -DNDEBUG -O2 -Wall -fPIC -O2   -isystem /home/sagemaker-user/.conda/envs/myenv2/include -fPIC -O2   -isystem /home/sagemaker-user/.conda/envs/myenv2/include 
BASECFLAGS:   -fno-strict-overflow -Wsign-compare
OPT:          -DNDEBUG -O2 -Wall
CONFIG_ARGS:  '--prefix=/home/sagemaker-user/.conda/envs/myenv2''--build=x86_64-conda-linux-gnu''--host=x86_64-conda-linux-gnu''--enable-ipv6''--with-ensurepip=no''--with-tzpath=/home/sagemaker-user/.conda/envs/myenv2/share/zoneinfo:/home/sagemaker-user/.conda/envs/myenv2/share/tzinfo''--with-computed-gotos''--with-system-expat''--enable-loadable-sqlite-extensions''--with-tcltk-includes=-I/home/sagemaker-user/.conda/envs/myenv2/include''--with-tcltk-libs=-L/home/sagemaker-user/.conda/envs/myenv2/lib -ltcl8.6 -ltk8.6''--with-platlibdir=lib''--with-lto=full''--enable-optimizations''-oldincludedir=/home/conda/feedstock_root/build_artifacts/python-split_

In [6]:
ds = load_dataset("openai/gsm8k", "main")

In [8]:
def preprocess_dataset(ds, usage, data_sampling_ratio=1, seed=125):
    sample_cnt = int(len(ds[usage]["answer"]) * data_sampling_ratio)
    random.seed(seed)
    sampled_data_idx = random.sample(range(0, len(ds[usage]["answer"])), sample_cnt)
    getter = operator.itemgetter(*sampled_data_idx)
    questions, answers =  getter(ds[usage]["question"]), getter(ds[usage]["answer"])
    print(len(questions), len(answers))
    with open("prompts/r1_zero.prompt", "r", encoding="utf-8") as f:
        prompt_string = f.read()

    def process_question(q):
        return prompt_string.format(question=q)
    def process_ground_truth(ans):
        return ans.split('\n#### ')[1]
    def process_prompt_completion(q, ans):
        prompt = prompt_string.format(question=q)
        cot =' ' + ans.split('\n#### ')[0] + ' </think>'
        gt = f" <answer> {ans.split('\n#### ')[1]} </answer>"
        return prompt + cot + gt
    with ThreadPoolExecutor() as executor:
        question_prompts = list(executor.map(process_question, ds[usage]["question"]))
    with ThreadPoolExecutor() as executor:
        ground_truth = list(executor.map(process_ground_truth, ds[usage]["answer"]))
    with ThreadPoolExecutor() as executor:
        prompt_completion = list(executor.map(process_prompt_completion, ds[usage]["question"], ds[usage]["answer"]))
    return question_prompts, ground_truth, prompt_completion

training_question_prompt, training_gt, training_data = preprocess_dataset(ds, 'train')
test_prompt, test_gt =  preprocess_dataset(ds, 'test')[0], preprocess_dataset(ds, 'test')[1]

train_ds = Dataset.from_dict({
    "data_source": ["train"] *len(training_question_prompt),
    "prompt":  training_question_prompt,
    "ground_truth":training_gt
    })
train_ds.to_parquet("dataset/train.parquet")

val_ds = Dataset.from_dict({
    "data_source": ["validation"]*len(test_prompt[:len(test_prompt)//2]),
    "prompt": test_prompt[:len(test_prompt)//2],
    "ground_truth": test_gt[:len(test_gt)//2],

})
val_ds.to_parquet("dataset/val.parquet")



7473 7473
1319 1319
1319 1319


Creating parquet from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

460514

In [1]:
import os
os.environ["WANDB_API_KEY"] = "860a38aad4d2008cc8519ea330be6f86ac1222b0"

In [2]:
%%bash
set -x

# If you are using vllm<=0.6.3, you might need to set the following environment variable to avoid bugs:
# export VLLM_ATTENTION_BACKEND=XFORMERS

train_path=dataset/train.parquet
val_path=dataset/val.parquet

train_files="['$train_path']"
test_files="['$val_path']"

python3 -m verl.trainer.main_ppo \
    algorithm.adv_estimator=grpo \
    data.train_files="$train_files" \
    data.val_files="$test_files" \
    data.train_batch_size=128 \
    data.max_prompt_length=1024 \
    data.max_response_length=1024 \
    data.filter_overlong_prompts=True \
    data.truncation='error' \
    actor_rollout_ref.model.path="model/sft_iter1_merged_model" \
    actor_rollout_ref.actor.optim.lr=1e-6 \
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.actor.strategy=fsdp \
    actor_rollout_ref.actor.ppo_epochs=3 \
    actor_rollout_ref.actor.ppo_mini_batch_size=128 \
    actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu=1 \
    actor_rollout_ref.actor.use_kl_loss=True \
    actor_rollout_ref.actor.kl_loss_coef=0.001 \
    actor_rollout_ref.actor.kl_loss_type=low_var_kl \
    actor_rollout_ref.actor.entropy_coeff=0 \
    actor_rollout_ref.model.enable_gradient_checkpointing=True \
    actor_rollout_ref.actor.fsdp_config.param_offload=True \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=True \
    actor_rollout_ref.rollout.log_prob_micro_batch_size_per_gpu=1 \
    actor_rollout_ref.rollout.tensor_model_parallel_size=1 \
    actor_rollout_ref.rollout.name=vllm \
    actor_rollout_ref.rollout.temperature=1.0 \
    actor_rollout_ref.rollout.top_k=-1 \
    actor_rollout_ref.rollout.top_p=1 \
    actor_rollout_ref.rollout.dtype=bfloat16 \
    actor_rollout_ref.rollout.gpu_memory_utilization=0.7 \
    actor_rollout_ref.rollout.n=5 \
    actor_rollout_ref.ref.log_prob_micro_batch_size_per_gpu=1 \
    actor_rollout_ref.ref.fsdp_config.param_offload=True \
    reward_model.reward_manager=CustomRewardManager \
    algorithm.use_kl_in_reward=False \
    trainer.critic_warmup=0 \
    trainer.logger=['console','wandb'] \
    trainer.project_name='verl_grpo_math_first_project' \
    trainer.experiment_name='my_first_math_experiment' \
    trainer.n_gpus_per_node=1 \
    trainer.nnodes=1 \
    trainer.save_freq=20 \
    trainer.test_freq=5 \
    trainer.total_epochs=3 $@

+ train_path=dataset/train.parquet
+ val_path=dataset/val.parquet
+ train_files='['\''dataset/train.parquet'\'']'
+ test_files='['\''dataset/val.parquet'\'']'
+ python3 -m verl.trainer.main_ppo algorithm.adv_estimator=grpo 'data.train_files=['\''dataset/train.parquet'\'']' 'data.val_files=['\''dataset/val.parquet'\'']' data.train_batch_size=128 data.max_prompt_length=1024 data.max_response_length=1024 data.filter_overlong_prompts=True data.truncation=error actor_rollout_ref.model.path=model/sft_iter1_merged_model actor_rollout_ref.actor.optim.lr=1e-6 actor_rollout_ref.model.use_remove_padding=True actor_rollout_ref.actor.strategy=fsdp actor_rollout_ref.actor.ppo_epochs=3 actor_rollout_ref.actor.ppo_mini_batch_size=128 actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu=1 actor_rollout_ref.actor.use_kl_loss=True actor_rollout_ref.actor.kl_loss_coef=0.001 actor_rollout_ref.actor.kl_loss_type=low_var_kl actor_rollout_ref.actor.entropy_coeff=0 actor_rollout_ref.model.enable_gradient_check

(TaskRunner pid=6578) TaskRunner hostname: default, PID: 6578
(TaskRunner pid=6578) {'actor_rollout_ref': {'actor': {'checkpoint': {'load_contents': ['model',
(TaskRunner pid=6578)                                                                   'optimizer',
(TaskRunner pid=6578)                                                                   'extra'],
(TaskRunner pid=6578)                                                 'save_contents': ['model',
(TaskRunner pid=6578)                                                                   'optimizer',
(TaskRunner pid=6578)                                                                   'extra']},
(TaskRunner pid=6578)                                  'clip_ratio': 0.2,
(TaskRunner pid=6578)                                  'clip_ratio_c': 3.0,
(TaskRunner pid=6578)                                  'clip_ratio_high': 0.2,
(TaskRunner pid=6578)                                  'clip_ratio_low': 0.2,
(TaskRunner pid=6578)                 

Filtering prompts longer than 1024 tokens:  94%|██████��██▎| 7000/7473 [00:27<00:01, 255.98 examples/s]


(TaskRunner pid=6578) filter dataset len: 7473
(TaskRunner pid=6578) Using dataset class: RLHFDataset


Filtering prompts longer than 1024 tokens: 100%|██████��███| 7473/7473 [00:29<00:00, 254.68 examples/s]0%|██████████| 7473/7473 [00:29<00:00, 255.12 examples/s]


...

(TaskRunner pid=6578) Checkpoint tracker file does not exist: /home/sagemaker-user/assignment5-alignment/cs336_alignment/checkpoints/verl_grpo_math_first_project/my_first_math_experiment/latest_checkpointed_iteration.txt
(TaskRunner pid=6578) Training from scratch
(TaskRunner pid=6578) test_gen_batch meta info: {'eos_token_id': 151645, 'pad_token_id': 151665, 'recompute_log_prob': False, 'do_sample': False, 'validate': True}


(WorkerDict pid=6854) /home/sagemaker-user/.conda/envs/myenv2/lib/python3.12/site-packages/torch/distributed/fsdp/_state_dict_utils.py:773: UserWarning: When using ``NO_SHARD`` for ``ShardingStrategy``, full_state_dict willbe returned.
(WorkerDict pid=6854)   warnings.warn(
(WorkerDict pid=6854) /home/sagemaker-user/.conda/envs/myenv2/lib/python3.12/site-packages/torch/distributed/fsdp/_state_dict_utils.py:711: UserWarning: When using ``NO_SHARD`` for ``ShardingStrategy``, full_state_dict willbe returned.
(WorkerDict pid=6854)   warnings.warn(


(TaskRunner pid=6578) validation generation end
(TaskRunner pid=6578) len reward_extra_infos_dict['reward']: 659
(TaskRunner pid=6578) ("Initial validation metrics: {'val-core/validation/reward/mean@659': "
(TaskRunner pid=6578)  "np.float64(0.0), 'val-aux/validation/reward/std@659': np.float64(0.0), "
(TaskRunner pid=6578)  "'val-aux/validation/reward/best@2/mean': np.float64(0.0), "
(TaskRunner pid=6578)  "'val-aux/validation/reward/best@2/std': np.float64(0.0), "
(TaskRunner pid=6578)  "'val-aux/validation/reward/worst@2/mean': np.float64(0.0), "
(TaskRunner pid=6578)  "'val-aux/validation/reward/worst@2/std': np.float64(0.0), "
(TaskRunner pid=6578)  "'val-aux/validation/reward/best@4/mean': np.float64(0.0), "
(TaskRunner pid=6578)  "'val-aux/validation/reward/best@4/std': np.float64(0.0), "
(TaskRunner pid=6578)  "'val-aux/validation/reward/worst@4/mean': np.float64(0.0), "
(TaskRunner pid=6578)  "'val-aux/validation/reward/worst@4/std': np.float64(0.0), "
(TaskRunner pid=6578)  "

Training Progress:   0%|          | 0/174 [00:00<?, ?it/s]
Error executing job with overrides: ['algorithm.adv_estimator=grpo', "data.train_files=['dataset/train.parquet']", "data.val_files=['dataset/val.parquet']", 'data.train_batch_size=128', 'data.max_prompt_length=1024', 'data.max_response_length=1024', 'data.filter_overlong_prompts=True', 'data.truncation=error', 'actor_rollout_ref.model.path=model/sft_iter1_merged_model', 'actor_rollout_ref.actor.optim.lr=1e-6', 'actor_rollout_ref.model.use_remove_padding=True', 'actor_rollout_ref.actor.strategy=fsdp', 'actor_rollout_ref.actor.ppo_epochs=3', 'actor_rollout_ref.actor.ppo_mini_batch_size=128', 'actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu=1', 'actor_rollout_ref.actor.use_kl_loss=True', 'actor_rollout_ref.actor.kl_loss_coef=0.001', 'actor_rollout_ref.actor.kl_loss_type=low_var_kl', 'actor_rollout_ref.actor.entropy_coeff=0', 'actor_rollout_ref.model.enable_gradient_checkpointing=True', 'actor_rollout_ref.actor.fsdp_config.par

CalledProcessError: Command 'b'set -x\n\n# If you are using vllm<=0.6.3, you might need to set the following environment variable to avoid bugs:\n# export VLLM_ATTENTION_BACKEND=XFORMERS\n\ntrain_path=dataset/train.parquet\nval_path=dataset/val.parquet\n\ntrain_files="[\'$train_path\']"\ntest_files="[\'$val_path\']"\n\npython3 -m verl.trainer.main_ppo \\\n    algorithm.adv_estimator=grpo \\\n    data.train_files="$train_files" \\\n    data.val_files="$test_files" \\\n    data.train_batch_size=128 \\\n    data.max_prompt_length=1024 \\\n    data.max_response_length=1024 \\\n    data.filter_overlong_prompts=True \\\n    data.truncation=\'error\' \\\n    actor_rollout_ref.model.path="model/sft_iter1_merged_model" \\\n    actor_rollout_ref.actor.optim.lr=1e-6 \\\n    actor_rollout_ref.model.use_remove_padding=True \\\n    actor_rollout_ref.actor.strategy=fsdp \\\n    actor_rollout_ref.actor.ppo_epochs=3 \\\n    actor_rollout_ref.actor.ppo_mini_batch_size=128 \\\n    actor_rollout_ref.actor

In [ ]:
# aws s3 cp sft_iter1_merged_model.tar.gz s3://588082972882-tryout/model/sft_iter1_merged_model.tar.gz

# aws s3 cp s3://588082972882-tryout/model/sft_iter1_merged_model.tar.gz .